# 03 – Candidate ligand–receptor pairs for the two directions

**Question.** Which ligand–receptor (L–R) pairs from the authors' CellPhoneDB table (S5) are candidates for
run A (**stroma → early lymphoid**) and run B (**early lymphoid → stroma**) in our cells?

Steps:
1. orient every S5 pair as ligand → receptor, using the CellPhoneDB v2 database flags, plus a few documented manual
   corrections;
2. keep a pair for a direction if the ligand is expressed in a sender population and the receptor in a receiver
   population (same 10% threshold as CellPhoneDB);
3. compare with the S5 values (read from the correctly oriented column) and with the paper's claims.

Populations and cells come from notebook 02 (`selected_cells.csv`). The per-population cap used for the CausalCCC input
does not matter here, because expression fractions are computed per population.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
DB = THY / "cellphonedb_v2_db/db_2019-09"
OUT = ROOT / "results/thymus_early/03_ligand_receptor_candidates"
OUT.mkdir(parents=True, exist_ok=True)

STROMA = ["cTEC", "mTEC(II)", "Fb_1", "Endo"]
LYMPHOID = ["ETP", "DN(early)", "DN(P)", "DN(Q)", "ILC3", "NK", "γδT"]

s5 = pd.read_csv(THY / "park2020_supplement/aay3224_table-s5.csv", header=1, low_memory=False).drop(columns="Unnamed: 0")
prot = pd.read_csv(DB / "protein_input.csv").set_index("uniprot")
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
# The count matrix uses older names for the nectins (PVRL1-3); rename them to match
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
print("S5 pairs:", len(s5))

S5 pairs: 710


## 1. Partners → genes, and receptor flags

A partner is either a single protein (`simple:<UniProt>`) or a complex (`complex:<name>`). For each partner we record
its genes and two database flags: `receptor` and `integrin`.

Genes are matched by **gene symbol**, not Ensembl ID. The database lists several Ensembl IDs for some proteins (the
main gene plus copies on alternative genome assemblies, mostly in the MHC region), and only the main one is in our
count matrix. Matching by Ensembl ID would silently lose LTB, LTA, TNF, DLL1, NOTCH4, B2M, PTPRC and others.

In [2]:
def partner_info(p):
    kind, name = p.split(":", 1)
    if kind == "simple":
        if name not in prot.index:
            return dict(genes=None, receptor=None, integrin=None)
        return dict(genes=[gene.gene_name.get(name)], receptor=bool(prot.receptor[name]),
                    integrin=bool(prot.integrin[name]))
    row = cplx.loc[name]
    units = [u for u in row[["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()]
    return dict(genes=[gene.gene_name.get(u) for u in units], receptor=bool(row.receptor), integrin=bool(row.integrin))

rows = []
for _, r in s5.iterrows():
    a, b = partner_info(r.partner_a), partner_info(r.partner_b)
    rows.append(dict(pair=r.interacting_pair, partner_a=r.partner_a, partner_b=r.partner_b,
                     genes_a=a["genes"], genes_b=b["genes"], receptor_a=a["receptor"], receptor_b=b["receptor"],
                     integrin_a=a["integrin"], integrin_b=b["integrin"]))
lr = pd.DataFrame(rows)
print("pairs with a partner missing from the database:", lr.genes_a.isna().sum() + lr.genes_b.isna().sum(),
      lr.loc[lr.genes_a.isna() | lr.genes_b.isna(), "pair"].tolist())
lr = lr.dropna(subset=["genes_a", "genes_b"]).reset_index(drop=True)

pairs with a partner missing from the database: 1 ['RIF1_NOTCH1']


## 2. Orientation

Rules, applied in order:
1. **Manual ligands.** Some membrane-anchored ligands are flagged `receptor` in the database because they are
   transmembrane proteins: TNFSF11 (RANKL), CXCL16, HBEGF, NRG1, SEMA4C, SEMA4D. In any pair containing one of them,
   that gene is the ligand. (EFNB2/EFNB3, the ephrin-B ligands, also carry the flag, but ephrin-B/EphB signalling
   genuinely goes both ways, so they are left to the rules below.)
2. **Receptor flag.** If exactly one partner is flagged `receptor`, the other is the ligand.
3. **Integrin.** If neither or both are flagged, and exactly one partner is an integrin, the integrin is the receptor.
4. Otherwise the pair is **ambiguous** (e.g. adhesion molecules binding each other, EGFR family members).
   Ambiguous pairs are kept **in both orientations**, flagged, and the expression filter decides.

In [3]:
MANUAL_LIGANDS = {"TNFSF11", "CXCL16", "HBEGF", "NRG1", "SEMA4C", "SEMA4D"}
def orient(r):
    sa = set(r.genes_a) if r.partner_a.startswith("simple") else set()
    sb = set(r.genes_b) if r.partner_b.startswith("simple") else set()
    if sa & MANUAL_LIGANDS and not sb & MANUAL_LIGANDS: return "a", "manual"
    if sb & MANUAL_LIGANDS and not sa & MANUAL_LIGANDS: return "b", "manual"
    if r.receptor_a != r.receptor_b:                    return ("b" if r.receptor_a else "a"), "receptor flag"
    if r.integrin_a != r.integrin_b:                    return ("b" if r.integrin_a else "a"), "integrin"
    return None, "ambiguous"

lr[["ligand_side", "orientation_rule"]] = lr.apply(lambda r: pd.Series(orient(r)), axis=1)
lr.orientation_rule.value_counts()

orientation_rule
receptor flag    431
integrin         208
ambiguous         55
manual            15
Name: count, dtype: int64

In [4]:
# One row per oriented candidate: ambiguous pairs appear twice (a -> b and b -> a)
def as_oriented(r, side):
    lig, rec = ("a", "b") if side == "a" else ("b", "a")
    return dict(pair=r.pair, ligand=r["partner_" + lig].split(":", 1)[1], receptor=r["partner_" + rec].split(":", 1)[1],
                ligand_genes=r["genes_" + lig], receptor_genes=r["genes_" + rec],
                ligand_is_partner_a=(lig == "a"), orientation_rule=r.orientation_rule)

oriented = pd.DataFrame([as_oriented(r, s) for _, r in lr.iterrows()
                         for s in ([r.ligand_side] if r.ligand_side else ["a", "b"])])
# readable names: gene symbol for single proteins, complex name otherwise
oriented["ligand"] = [g[0] if l in gene.index else l for l, g in zip(oriented.ligand, oriented.ligand_genes)]
oriented["receptor"] = [g[0] if l in gene.index else l for l, g in zip(oriented.receptor, oriented.receptor_genes)]
print(len(oriented), "oriented candidates from", len(lr), "S5 pairs")
oriented[oriented.pair.isin(["FGFR2_FGF10", "TNFSF11_TNFRSF11A", "JAM2_JAM3", "CD40LG_a5b1 complex"])][
    ["pair", "ligand", "receptor", "ligand_is_partner_a", "orientation_rule"]]

709 oriented candidates from 709 S5 pairs


,pair,ligand,receptor,ligand_is_partner_a,orientation_rule
222,JAM2_JAM3,JAM3,JAM2,False,ambiguous
258,FGFR2_FGF10,FGF10,FGFR2,False,receptor flag
357,TNFSF11_TNFRSF11A,TNFSF11,TNFRSF11A,True,manual
608,CD40LG_a5b1 complex,CD40LG,a5b1 complex,True,integrin


## 3. Expression in our populations

We read raw counts for the selected cells and for the genes involved in S5 only, and compute, per population, the
fraction of cells with at least one count. For a complex, the fraction is the **minimum over its subunits**
(all subunits must be present, as in CellPhoneDB). A side "passes" in a population if the fraction is ≥ 10%.

In [5]:
cells = pd.read_csv(ROOT / "results/thymus_early/02_cell_populations/selected_cells.csv", index_col=0)
raw = ad.read_h5ad(THY / "zenodo_5500511/HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")

all_genes = sorted({g for gs in pd.concat([oriented.ligand_genes, oriented.receptor_genes]) for g in gs if g})
gene_pos = pd.Series(np.arange(raw.n_vars), index=raw.var_names)   # var_names are gene symbols
missing = [g for g in all_genes if g not in gene_pos.index]
print("S5 genes:", len(all_genes), "| not in the count matrix:", missing)
all_genes = [g for g in all_genes if g in gene_pos.index]
# 34 gene symbols occur twice in the matrix; none of them is an S5 gene, so each lookup is unambiguous
assert not set(all_genes) & set(raw.var_names[raw.var_names.duplicated()])

row_pos = np.sort(raw.obs_names.get_indexer(cells.index))
X = raw.X[row_pos][:, gene_pos[all_genes].values]          # cells x S5 genes, raw counts
pop_of_row = cells.loc[raw.obs_names[row_pos], "population"].values

detected = pd.DataFrame((X > 0).toarray(), columns=all_genes)
frac = detected.groupby(pop_of_row).mean()                 # populations x genes
frac.shape

/tmp/ipykernel_263068/2527484340.py:2: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "zenodo_5500511/HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


S5 genes: 516 | not in the count matrix: []


(11, 516)

In [6]:
THRESHOLD = 0.10

def side_fraction(genes, pops):
    genes = [g for g in genes if g in frac.columns]
    if not genes:
        return pd.Series(0.0, index=pops)
    return frac.loc[pops, genes].min(axis=1)               # complex: weakest subunit

def candidates(senders, receivers, name):
    out = []
    for _, r in oriented.iterrows():
        fl, fr = side_fraction(r.ligand_genes, senders), side_fraction(r.receptor_genes, receivers)
        if fl.max() >= THRESHOLD and fr.max() >= THRESHOLD:
            out.append(dict(direction=name, pair=r.pair, ligand=r.ligand, receptor=r.receptor,
                            orientation_rule=r.orientation_rule, ligand_is_partner_a=r.ligand_is_partner_a,
                            max_ligand_frac=fl.max(), ligand_pops=", ".join(fl.index[fl >= THRESHOLD]),
                            max_receptor_frac=fr.max(), receptor_pops=", ".join(fr.index[fr >= THRESHOLD])))
    return pd.DataFrame(out)

cand_A = candidates(STROMA, LYMPHOID, "A: stroma -> lymphoid")
cand_B = candidates(LYMPHOID, STROMA, "B: lymphoid -> stroma")
print("run A candidates:", len(cand_A), "| run B candidates:", len(cand_B))
pd.concat([cand_A, cand_B]).groupby(["direction", "orientation_rule"]).size().unstack(fill_value=0)

run A candidates: 132 | run B candidates: 156


orientation_rule,ambiguous,integrin,manual,receptor flag
direction,,,,
A: stroma -> lymphoid,14,16,3,99
B: lymphoid -> stroma,14,18,5,119


## 4. Comparison with the S5 values

S5 column `X_Y` means population X expresses `partner_a` and population Y expresses `partner_b` (checked in the
discussion of notebook 02's plan: e.g. `CCL25_CCR9` is 1.45 in `cTEC_DN(early)` but 0.02 in `DN(early)_cTEC`).
So the sender → receiver value is in `sender_receiver` when the ligand is partner_a, and in `receiver_sender`
otherwise. Our population names are mapped to the S5 labels; S5 merges the mTEC subtypes into `mTEC`, so mTEC(II)
uses `mTEC`. S5 was computed on an earlier annotation (notebook 01), so this is a rough comparison.

In [7]:
TO_S5 = {"mTEC(II)": "mTEC", "γδT": "gdT"}
s5v = s5.set_index("interacting_pair")

def s5_best(r, senders, receivers):
    best = (0.0, "")
    for s in senders:
        for t in receivers:
            S, T = TO_S5.get(s, s), TO_S5.get(t, t)
            col = f"{S}_{T}" if r.ligand_is_partner_a else f"{T}_{S}"
            v = float(s5v.loc[r.pair, col])
            if v > best[0]:
                best = (v, f"{s} -> {t}")
    return pd.Series(best, index=["s5_max_mean", "s5_best_pair"])

cand_A[["s5_max_mean", "s5_best_pair"]] = cand_A.apply(s5_best, axis=1, senders=STROMA, receivers=LYMPHOID)
cand_B[["s5_max_mean", "s5_best_pair"]] = cand_B.apply(s5_best, axis=1, senders=LYMPHOID, receivers=STROMA)

show = ["pair", "ligand", "receptor", "orientation_rule", "max_ligand_frac", "ligand_pops",
        "max_receptor_frac", "receptor_pops", "s5_max_mean", "s5_best_pair"]
cand_A.sort_values("s5_max_mean", ascending=False)[show].head(30).round(2)

,pair,ligand,receptor,orientation_rule,max_ligand_frac,ligand_pops,max_receptor_frac,receptor_pops,s5_max_mean,s5_best_pair
53,CD74_MIF,MIF,CD74,receptor flag,0.94,"cTEC, mTEC(II), Fb_1, Endo",0.74,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",2.38,cTEC -> ETP
7,CCL19_CCR7,CCL19,CCR7,receptor flag,0.93,"cTEC, mTEC(II)",0.18,"ETP, NK, γδT",1.99,mTEC(II) -> γδT
20,APP_CD74,APP,CD74,receptor flag,0.89,"cTEC, mTEC(II), Fb_1, Endo",0.74,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",1.99,Endo -> ETP
1,CCL19_CXCR3,CCL19,CXCR3,receptor flag,0.93,"cTEC, mTEC(II)",0.24,NK,1.96,mTEC(II) -> NK
80,IGF1R_IGF2,IGF2,IGF1R,receptor flag,1.00,"mTEC(II), Fb_1, Endo",0.41,"ETP, DN(early), DN(P), DN(Q), ILC3, γδT",1.86,Fb_1 -> DN(P)
64,IGF2_IGF2R,IGF2,IGF2R,receptor flag,1.00,"mTEC(II), Fb_1, Endo",0.27,"DN(early), DN(P), DN(Q), NK, γδT",1.86,Fb_1 -> NK
130,CD94:NKG2A_HLAE_HUMAN,HLA-E,CD94:NKG2A,receptor flag,0.93,"cTEC, mTEC(II), Fb_1, Endo",0.76,NK,1.84,Endo -> NK
98,HLAE_HUMAN_KLRC1,HLA-E,KLRC1,receptor flag,0.93,"cTEC, mTEC(II), Fb_1, Endo",0.81,"ILC3, NK",1.84,Endo -> NK
9,CCL25_CCR9,CCL25,CCR9,receptor flag,0.98,"cTEC, mTEC(II)",0.37,"ETP, DN(Q), ILC3, γδT",1.81,cTEC -> γδT
8,IGF2_IDE,IGF2,IDE,receptor flag,1.00,"mTEC(II), Fb_1, Endo",0.20,"ETP, DN(early), DN(P), γδT",1.78,Fb_1 -> γδT


In [8]:
cand_B.sort_values("s5_max_mean", ascending=False)[show].head(30).round(2)

,pair,ligand,receptor,orientation_rule,max_ligand_frac,ligand_pops,max_receptor_frac,receptor_pops,s5_max_mean,s5_best_pair
63,CD74_MIF,MIF,CD74,receptor flag,0.97,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.97,"cTEC, mTEC(II), Fb_1, Endo",2.59,DN(early) -> mTEC(II)
42,LTB_LTBR,LTB,LTBR,receptor flag,0.94,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.45,"cTEC, mTEC(II), Fb_1, Endo",2.04,ILC3 -> Fb_1
6,ACKR1_CCL5,CCL5,ACKR1,receptor flag,0.36,"ILC3, NK",0.52,"mTEC(II), Endo",1.99,NK -> Endo
112,CD99_PILRA,CD99,PILRA,receptor flag,1.00,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.22,mTEC(II),1.84,DN(Q) -> mTEC(II)
15,APP_CD74,APP,CD74,receptor flag,0.65,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.97,"cTEC, mTEC(II), Fb_1, Endo",1.76,ETP -> mTEC(II)
46,PECAM1_CD38,CD38,PECAM1,ambiguous,0.50,"ETP, DN(early), DN(P), DN(Q), γδT",0.97,Endo,1.37,NK -> Endo
44,MIF_TNFRSF14,MIF,TNFRSF14,receptor flag,0.97,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.61,"cTEC, mTEC(II), Endo",1.29,DN(early) -> mTEC(II)
61,MIF_TNFRSF10D,MIF,TNFRSF10D,receptor flag,0.97,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.47,"cTEC, mTEC(II), Endo",1.25,DN(early) -> Endo
118,CLEC2B_KLRF1,KLRF1,CLEC2B,ambiguous,0.33,NK,0.26,Endo,1.13,NK -> Endo
7,ACKR1_CXCL1,CXCL1,ACKR1,receptor flag,0.11,ILC3,0.52,"mTEC(II), Endo",0.97,ILC3 -> Endo


## 5. The paper's early-niche claims

Where do the claimed pairs end up? A claim can be absent from both runs for two reasons: it is not a stroma ↔ lymphoid
interaction (FGF7 → FGFR2 is fibroblast → TEC), or it fails the 10% expression filter in our cells.

In [9]:
CLAIMS = {
    "LTB_LTBR": "LTB from immune cells -> LTBR on most stromal cells",
    "CD40LG_CD40": "in the authors' early-stage dot plot",
    "TNFSF11_TNFRSF11A": "RANKL-RANK, ILC3 <-> mTEC(II)",
    "FGF7_FGFR2": "fibroblasts -> TECs",
    "JAG1_NOTCH1": "other TECs express JAG1",
    "JAG2_NOTCH1": "cTEC and Endo express JAG2; NOTCH1 on ETP",
    "DLL4_NOTCH1": "cTEC and Endo express DLL4; NOTCH1 on ETP",
    "DLL1_NOTCH1": "in the authors' early-stage dot plot",
}
both = pd.concat([cand_A, cand_B])
claim_rows = []
for pair, text in CLAIMS.items():
    hit = both[both.pair == pair]
    o = oriented[oriented.pair == pair].iloc[0]
    claim_rows.append(dict(pair=pair, claim=text, oriented_as=f"{o.ligand} -> {o.receptor}",
                           in_run_A=(hit.direction.str.startswith("A")).any(),
                           in_run_B=(hit.direction.str.startswith("B")).any()))
claims = pd.DataFrame(claim_rows)
claims

,pair,claim,oriented_as,in_run_A,in_run_B
0,LTB_LTBR,LTB from immune cells -> LTBR on most stromal ...,LTB -> LTBR,True,True
1,CD40LG_CD40,in the authors' early-stage dot plot,CD40LG -> CD40,False,True
2,TNFSF11_TNFRSF11A,"RANKL-RANK, ILC3 <-> mTEC(II)",TNFSF11 -> TNFRSF11A,False,True
3,FGF7_FGFR2,fibroblasts -> TECs,FGF7 -> FGFR2,False,False
4,JAG1_NOTCH1,other TECs express JAG1,JAG1 -> NOTCH1,True,True
5,JAG2_NOTCH1,cTEC and Endo express JAG2; NOTCH1 on ETP,JAG2 -> NOTCH1,True,False
6,DLL4_NOTCH1,cTEC and Endo express DLL4; NOTCH1 on ETP,DLL4 -> NOTCH1,True,False
7,DLL1_NOTCH1,in the authors' early-stage dot plot,DLL1 -> NOTCH1,True,True


In [10]:
# Expression fractions behind the claims, in every kept population
claim_genes = ["LTB", "LTBR", "CD40LG", "CD40", "TNFSF11", "TNFRSF11A", "FGF7", "FGFR2",
               "JAG1", "JAG2", "DLL1", "DLL4", "NOTCH1", "NOTCH2", "NOTCH3"]
frac.loc[STROMA + LYMPHOID, claim_genes].round(2)

,LTB,LTBR,CD40LG,CD40,TNFSF11,TNFRSF11A,FGF7,FGFR2,JAG1,JAG2,DLL1,DLL4,NOTCH1,NOTCH2,NOTCH3
cTEC,0.11,0.28,0.00,0.47,0.00,0.17,0.09,0.29,0.18,0.56,0.07,0.34,0.33,0.44,0.21
mTEC(II),0.50,0.20,0.00,0.69,0.00,0.54,0.04,0.29,0.26,0.20,0.06,0.05,0.43,0.35,0.23
Fb_1,0.16,0.44,0.00,0.02,0.00,0.00,0.76,0.00,0.09,0.00,0.06,0.01,0.09,0.44,0.14
Endo,0.30,0.45,0.00,0.63,0.00,0.06,0.03,0.00,0.28,0.39,0.25,0.49,0.52,0.03,0.09
ETP,0.89,0.13,0.00,0.03,0.01,0.01,0.01,0.01,0.13,0.04,0.07,0.02,0.27,0.08,0.03
DN(early),0.72,0.21,0.00,0.01,0.00,0.00,0.02,0.00,0.06,0.01,0.03,0.08,0.58,0.14,0.20
DN(P),0.62,0.04,0.00,0.01,0.01,0.00,0.02,0.00,0.07,0.01,0.03,0.05,0.51,0.12,0.33
DN(Q),0.70,0.01,0.00,0.00,0.00,0.00,0.01,0.00,0.04,0.01,0.01,0.04,0.35,0.07,0.32
ILC3,0.94,0.03,0.13,0.02,0.26,0.06,0.01,0.00,0.28,0.01,0.35,0.02,0.04,0.11,0.01
NK,0.61,0.02,0.01,0.00,0.10,0.05,0.00,0.00,0.17,0.01,0.06,0.01,0.14,0.07,0.01


In [11]:
cand_A.to_csv(OUT / "candidates_A_stroma_to_lymphoid.csv", index=False)
cand_B.to_csv(OUT / "candidates_B_lymphoid_to_stroma.csv", index=False)
oriented.drop(columns=["ligand_genes", "receptor_genes"]).to_csv(OUT / "s5_oriented.csv", index=False)
frac.T.round(3).to_csv(OUT / "expression_fraction_by_population.csv")
print("written to", OUT)

written to ../../results/thymus_early/03_ligand_receptor_candidates


## Summary

| | |
|---|---|
| S5 pairs | 710; 1 dropped (`RIF1_NOTCH1`, partner not in the database) |
| Orientation | receptor flag 431 · integrin 208 · manual 15 · ambiguous 55 (kept in both orientations) |
| Gene matching | by symbol (Ensembl IDs would lose MHC-region genes such as LTB, TNF, DLL1, NOTCH4); nectins renamed to PVRL1–3 |
| Candidates, run A (stroma → lymphoid) | 132 oriented pairs pass the 10% filter |
| Candidates, run B (lymphoid → stroma) | 156 |

**The paper's claims in our cells** (fractions of expressing cells, section 5):
- **Notch:** JAG2 and DLL4 are expressed in cTEC (56%, 34%) and Endo (39%, 49%), and NOTCH1 in ETP (27%) and DN
  (35–58%). JAG2/DLL4 → NOTCH1 are run-A candidates, consistent with the paper.
- **LTB → LTBR:** LTB is expressed in every lymphoid population (61–94%) and LTBR in the stroma (20–45%): run B, as claimed.
  It also passes in run A (LTB in mTEC(II) and Endo, LTBR in DN(early)), which is a by-product of the 10% threshold.
- **RANKL → RANK:** TNFSF11 is expressed in ILC3 (26%) and NK (10%), TNFRSF11A in mTEC(II) (54%): run B, as claimed.
- **CD40LG → CD40:** CD40LG reaches 10% only in ILC3 (13%); CD40 is in cTEC, mTEC(II) and Endo. Run B.
- **FGF7 → FGFR2:** FGF7 in Fb_1 (76%), FGFR2 in cTEC and mTEC(II) (29%). This is stroma → stroma, so it is in neither
  run; it would need a separate Fb_1 → TEC run.

Next (notebook 04): build the CausalCCC input for run A – per-population cap, gene selection, and which of the
run-A candidate pairs to include (CausalCCC networks usually include about 20–30 L–R links).